# DDoS Detection Project - Kaggle-Style EDA

Notebook này dùng để hiển thị toàn bộ thông tin quan trọng của project: nguồn dữ liệu CICDDoS2019, cấu trúc train/test, phân bố nhãn, chất lượng dữ liệu, thống kê feature, correlation, kết quả huấn luyện, artifact mô hình và log live IPS.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_SRC = PROJECT_ROOT / "backend" / "src"
if str(BACKEND_SRC) not in sys.path:
    sys.path.insert(0, str(BACKEND_SRC))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "saved_models"
AUDIT_DIR = PROJECT_ROOT / "audit_results"

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)
print("Models dir:", MODELS_DIR)
print("Audit dir:", AUDIT_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, Markdown

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Nguồn dữ liệu

Dataset sử dụng trong project là **CICDDoS2019** từ Kaggle: `dhoogla/cicddos2019`. Dữ liệu trong project được đọc từ các file Parquet trong thư mục `data/`.

In [ ]:
from ml_ddos.data_loader import KAGGLE_DATASET_SLUG, collect_file_paths, load_dataset

print("Kaggle dataset slug:", KAGGLE_DATASET_SLUG)

train_paths, test_paths = collect_file_paths(str(DATA_DIR))
print("Training files:", len(train_paths))
print("Testing files:", len(test_paths))

display(pd.DataFrame({"training_file": [Path(p).name for p in train_paths]}).head(50))
display(pd.DataFrame({"testing_file": [Path(p).name for p in test_paths]}).head(50))

## 2. Load dữ liệu train/test

In [ ]:
train_df, test_df = load_dataset(str(DATA_DIR))

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

## 2.1. Hiển thị bộ dữ liệu thô theo phong cách Kaggle

Các cell dưới đây tập trung vào việc quan sát trực tiếp dữ liệu trước khi huấn luyện: kích thước, tên cột, dòng đầu/cuối, mẫu ngẫu nhiên và `info()`. Đây là bước EDA cơ bản để hiểu dữ liệu đầu vào trước khi tiền xử lý.

In [ ]:
import io

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:")
display(pd.DataFrame({"index": range(len(train_df.columns)), "column": train_df.columns}))

display(Markdown("### Train head"))
display(train_df.head(10))

display(Markdown("### Train tail"))
display(train_df.tail(10))

display(Markdown("### Train random sample"))
display(train_df.sample(min(10, len(train_df)), random_state=42))

display(Markdown("### Test head"))
display(test_df.head(10))

## 2.2. Kiểm tra `info()`, kiểu dữ liệu và tỷ lệ missing

Trong Kaggle notebook, phần này thường dùng để phát hiện cột sai kiểu dữ liệu, cột thiếu dữ liệu, cột hằng số hoặc cột không hữu ích.

In [ ]:
def capture_info(df: pd.DataFrame) -> str:
    buffer = io.StringIO()
    df.info(buf=buffer, memory_usage="deep")
    return buffer.getvalue()

print("TRAIN INFO")
print(capture_info(train_df))

print("TEST INFO")
print(capture_info(test_df))

raw_missing = pd.DataFrame({
    "column": train_df.columns,
    "train_missing": [train_df[c].isna().sum() for c in train_df.columns],
    "train_missing_percent": [train_df[c].isna().mean() * 100 for c in train_df.columns],
    "test_missing": [test_df[c].isna().sum() for c in test_df.columns],
    "test_missing_percent": [test_df[c].isna().mean() * 100 for c in test_df.columns],
    "train_unique": [train_df[c].nunique(dropna=False) for c in train_df.columns],
    "test_unique": [test_df[c].nunique(dropna=False) for c in test_df.columns],
})

display(raw_missing.sort_values("train_missing_percent", ascending=False).head(50))

## 2.3. Thống kê mô tả dữ liệu thô

Phần này tương ứng với các lệnh kiểu `describe()`, `value_counts()` và kiểm tra phân bố dữ liệu thường thấy trên Kaggle.

In [ ]:
display(Markdown("### Numeric describe - train"))
display(train_df.describe(include=[np.number]).T)

display(Markdown("### Object/category describe - train"))
display(train_df.describe(include=["object", "category"]).T)

display(Markdown("### Raw label counts - train"))
display(train_df["Label"].value_counts().rename_axis("Label").reset_index(name="count"))

display(Markdown("### Raw label counts - test"))
display(test_df["Label"].value_counts().rename_axis("Label").reset_index(name="count"))

## 2.4. Minh họa quy trình tiền xử lý dữ liệu

Lưu ý quan trọng: cell này dùng để **hiển thị và giải thích** các bước tiền xử lý. Trong pipeline huấn luyện thật, các bước impute, scale, encode, loại cột leakage và loại feature tương quan cao được đặt trong `sklearn Pipeline` và chỉ `fit` trên training fold để tránh data leakage.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from ml_ddos.preprocessor import harmonize_labels, remove_duplicates, TARGET_COL
from ml_ddos.models import TrafficFeaturePreprocessor

print("Target column:", TARGET_COL)

raw_label_summary = pd.DataFrame({
    "train_before": train_df[TARGET_COL].value_counts(),
    "test_before": test_df[TARGET_COL].value_counts(),
}).fillna(0).astype(int)

display(Markdown("### Label distribution before harmonization"))
display(raw_label_summary)

harmonized_train, harmonized_test = harmonize_labels(train_df, test_df)

harmonized_label_summary = pd.DataFrame({
    "train_after": harmonized_train[TARGET_COL].value_counts(),
    "test_after": harmonized_test[TARGET_COL].value_counts(),
}).fillna(0).astype(int)

display(Markdown("### Label distribution after harmonization"))
display(harmonized_label_summary)

print("Train rows before duplicate removal:", len(harmonized_train))
print("Test rows before duplicate removal:", len(harmonized_test))

train_clean = remove_duplicates(harmonized_train, "notebook training preview")
test_clean = remove_duplicates(harmonized_test, "notebook testing preview")

print("Train rows after duplicate removal:", len(train_clean))
print("Test rows after duplicate removal:", len(test_clean))

## 2.5. Tách feature/label và mã hóa nhãn

Model học máy không học trực tiếp chuỗi nhãn như `Benign`, `TCP SYN Flood`, vì vậy nhãn được mã hóa thành số bằng `LabelEncoder`.

In [ ]:
X_train_raw = train_clean.drop(columns=[TARGET_COL])
y_train_raw = train_clean[TARGET_COL]
X_test_raw = test_clean.drop(columns=[TARGET_COL])
y_test_raw = test_clean[TARGET_COL]

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train_raw)
y_test_encoded = label_encoder.transform(y_test_raw)

label_mapping = pd.DataFrame({
    "encoded_label": range(len(label_encoder.classes_)),
    "class_name": label_encoder.classes_,
})

display(label_mapping)
print("X_train_raw shape:", X_train_raw.shape)
print("y_train_encoded shape:", y_train_encoded.shape)
print("X_test_raw shape:", X_test_raw.shape)
print("y_test_encoded shape:", y_test_encoded.shape)

## 2.6. Minh họa preprocessor của project

`TrafficFeaturePreprocessor` thực hiện các bước chính:

- Loại bỏ cột nghi ngờ leakage như label, target, attack, flow id, timestamp, source/destination IP nếu có.
- Loại bỏ cột hằng số.
- Xử lý `NaN`/`infinity`.
- Impute numeric bằng median.
- Scale numeric bằng `MinMaxScaler`.
- Encode categorical bằng `OneHotEncoder`.
- Loại bỏ feature tương quan cao theo ngưỡng cấu hình.

Để notebook chạy nhẹ hơn, cell dưới đây fit preprocessor trên một mẫu tối đa 50.000 dòng của training set. Huấn luyện thật trong `main.py` vẫn dùng pipeline đầy đủ với cross-validation.

In [ ]:
MAX_PREPROCESS_ROWS = 50_000
if len(train_clean) > MAX_PREPROCESS_ROWS:
    try:
        preprocess_sample = train_clean.groupby(TARGET_COL, group_keys=False).sample(
            frac=MAX_PREPROCESS_ROWS / len(train_clean),
            random_state=42,
        )
    except ValueError:
        preprocess_sample = train_clean.sample(MAX_PREPROCESS_ROWS, random_state=42)
else:
    preprocess_sample = train_clean.copy()

X_pre_sample = preprocess_sample.drop(columns=[TARGET_COL])
y_pre_sample = label_encoder.transform(preprocess_sample[TARGET_COL])

preprocessor = TrafficFeaturePreprocessor(correlation_threshold=0.9)
X_transformed = preprocessor.fit_transform(X_pre_sample, y_pre_sample)

preprocess_report = pd.DataFrame({
    "item": [
        "original_columns",
        "leakage_columns_removed",
        "constant_columns_removed",
        "high_corr_columns_removed",
        "numeric_columns_after_filter",
        "categorical_columns_after_filter",
        "output_features",
        "sample_rows",
    ],
    "value": [
        len(preprocessor.original_columns_),
        len(preprocessor.leakage_columns_),
        len(preprocessor.constant_columns_),
        len(preprocessor.high_corr_columns_),
        len(preprocessor.numeric_columns_),
        len(preprocessor.categorical_columns_),
        X_transformed.shape[1],
        X_transformed.shape[0],
    ],
})

display(preprocess_report)

display(Markdown("### Leakage-like columns removed"))
display(pd.DataFrame({"column": preprocessor.leakage_columns_}))

display(Markdown("### Constant columns removed"))
display(pd.DataFrame({"column": preprocessor.constant_columns_}).head(50))

display(Markdown("### High-correlation columns removed"))
display(pd.DataFrame({"column": preprocessor.high_corr_columns_}).head(80))

print("Transformed matrix shape:", X_transformed.shape)
print("Transformed min:", np.nanmin(X_transformed))
print("Transformed max:", np.nanmax(X_transformed))

## 2.7. Minh họa chia validation theo stratify

Pipeline thật dùng cross-validation, không phụ thuộc vào một validation split duy nhất. Tuy nhiên, cell này minh họa cách chia stratified train/validation để báo cáo dễ hình dung.

In [ ]:
X_demo_train, X_demo_val, y_demo_train, y_demo_val = train_test_split(
    X_train_raw,
    y_train_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_train_encoded,
)

split_demo = pd.DataFrame({
    "split": ["train_demo", "validation_demo"],
    "rows": [len(X_demo_train), len(X_demo_val)],
    "features": [X_demo_train.shape[1], X_demo_val.shape[1]],
})

display(split_demo)

train_demo_dist = pd.Series(y_demo_train).value_counts(normalize=True).sort_index()
val_demo_dist = pd.Series(y_demo_val).value_counts(normalize=True).sort_index()
stratify_check = pd.DataFrame({
    "class_name": label_encoder.classes_,
    "train_demo_percent": train_demo_dist.reindex(range(len(label_encoder.classes_)), fill_value=0).values * 100,
    "validation_demo_percent": val_demo_dist.reindex(range(len(label_encoder.classes_)), fill_value=0).values * 100,
})

display(stratify_check)

## 3. Tổng quan cột, kiểu dữ liệu và bộ nhớ

In [ ]:
def dataframe_overview(df: pd.DataFrame, name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "dataset": name,
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [df[c].notna().sum() for c in df.columns],
        "null_count": [df[c].isna().sum() for c in df.columns],
        "null_percent": [df[c].isna().mean() * 100 for c in df.columns],
        "unique": [df[c].nunique(dropna=False) for c in df.columns],
    })

overview = pd.concat([
    dataframe_overview(train_df, "train"),
    dataframe_overview(test_df, "test"),
], ignore_index=True)

display(overview)
print("Train memory MB:", train_df.memory_usage(deep=True).sum() / 1024 / 1024)
print("Test memory MB:", test_df.memory_usage(deep=True).sum() / 1024 / 1024)

## 4. Phân bố nhãn

In [ ]:
TARGET_COL = "Label"

train_label_counts = train_df[TARGET_COL].value_counts().rename_axis("Label").reset_index(name="train_count")
test_label_counts = test_df[TARGET_COL].value_counts().rename_axis("Label").reset_index(name="test_count")
label_dist = train_label_counts.merge(test_label_counts, on="Label", how="outer").fillna(0)
label_dist["train_percent"] = label_dist["train_count"] / label_dist["train_count"].sum() * 100
label_dist["test_percent"] = label_dist["test_count"] / label_dist["test_count"].sum() * 100

display(label_dist.sort_values("train_count", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=label_dist.sort_values("train_count", ascending=False), x="train_count", y="Label", ax=axes[0])
axes[0].set_title("Training Label Distribution")
sns.barplot(data=label_dist.sort_values("test_count", ascending=False), x="test_count", y="Label", ax=axes[1])
axes[1].set_title("Testing Label Distribution")
plt.tight_layout()
plt.show()

## 5. Missing values, infinity và duplicate rows

In [ ]:
def quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    numeric = df.select_dtypes(include=[np.number])
    inf_count = np.isinf(numeric.to_numpy()).sum() if not numeric.empty else 0
    return pd.DataFrame([{
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "infinite_numeric_values": int(inf_count),
        "duplicate_rows": int(df.duplicated().sum()),
    }])

display(pd.concat([
    quality_report(train_df, "train"),
    quality_report(test_df, "test"),
], ignore_index=True))

missing_train = train_df.isna().sum().sort_values(ascending=False)
missing_train = missing_train[missing_train > 0].reset_index()
missing_train.columns = ["column", "missing_count"]
display(missing_train.head(50))

## 6. Thống kê numeric features

In [ ]:
numeric_cols = train_df.drop(columns=[TARGET_COL], errors="ignore").select_dtypes(include=[np.number]).columns.tolist()
print("Numeric feature count:", len(numeric_cols))
display(train_df[numeric_cols].describe().T.head(100))

## 7. Distribution shift giữa train và test

In [ ]:
shift_rows = []
for col in numeric_cols:
    train_s = pd.to_numeric(train_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    test_s = pd.to_numeric(test_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    train_mean = train_s.mean()
    test_mean = test_s.mean()
    train_std = train_s.std()
    normalized_gap = abs(train_mean - test_mean) / (train_std + 1e-9)
    shift_rows.append({
        "feature": col,
        "train_mean": train_mean,
        "test_mean": test_mean,
        "train_std": train_std,
        "normalized_mean_gap": normalized_gap,
    })

shift_df = pd.DataFrame(shift_rows).sort_values("normalized_mean_gap", ascending=False)
display(shift_df.head(30))

plt.figure(figsize=(12, 8))
sns.barplot(data=shift_df.head(20), x="normalized_mean_gap", y="feature")
plt.title("Top Train/Test Feature Distribution Gaps")
plt.tight_layout()
plt.show()

## 8. Correlation heatmap

In [ ]:
sample_for_corr = train_df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
variance_df = pd.DataFrame(
    {"feature": col, "variance": float(sample_for_corr[col].var())}
    for col in sample_for_corr.columns
)
top_var_cols = variance_df.sort_values("variance", ascending=False)["feature"].head(30).tolist()
corr = sample_for_corr[top_var_cols].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.2)
plt.title("Correlation Heatmap - Top Variance Numeric Features")
plt.tight_layout()
plt.show()

## 9. Kết quả huấn luyện và kiểm thử

In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    print("Missing:", path)
    return pd.DataFrame()

validation_scores = read_csv_if_exists(RESULTS_DIR / "validation_scores.csv")
test_scores = read_csv_if_exists(RESULTS_DIR / "test_scores.csv")
robust_summary = read_csv_if_exists(RESULTS_DIR / "robust_model_summary.csv")
per_class_metrics = read_csv_if_exists(RESULTS_DIR / "test_per_class_metrics.csv")

display(Markdown("### Cross-validation scores"))
display(validation_scores)
display(Markdown("### Test scores"))
display(test_scores)
display(Markdown("### Robust model summary"))
display(robust_summary)
display(Markdown("### Test per-class metrics"))
display(per_class_metrics.head(100))

In [ ]:
if not test_scores.empty and "Test F1-Score" in test_scores.columns:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=test_scores.sort_values("Test F1-Score", ascending=False), x="Test F1-Score", y="Model")
    plt.title("Test F1-Score by Model")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.show()

if not validation_scores.empty and "CV F1-Score Mean" in validation_scores.columns:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=validation_scores.sort_values("CV F1-Score Mean", ascending=False), x="CV F1-Score Mean", y="Model")
    plt.title("Cross-Validation F1-Score Mean by Model")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.show()

## 10. Hiển thị artifact hình ảnh trong `results/`

In [ ]:
image_files = [
    "model_comparison.png",
    "train_cv_test_f1_comparison.png",
    "selected_model_learning_curve.png",
    "extra_trees_test_confusion_matrix.png",
    "extra_trees_test_roc_curve.png",
    "extra_trees_feature_importance.png",
    "extra_trees_shap_summary_bar.png",
]

for image_name in image_files:
    image_path = RESULTS_DIR / image_name
    if image_path.exists():
        display(Markdown(f"### {image_name}"))
        display(Image(filename=str(image_path)))
    else:
        print("Missing image:", image_path)

## 11. Feature importance ranking

In [ ]:
feature_importance_files = sorted(RESULTS_DIR.glob("*_feature_importance.csv"))
print("Feature importance files:", [p.name for p in feature_importance_files])

for path in feature_importance_files:
    display(Markdown(f"### {path.name}"))
    fi_df = pd.read_csv(path)
    display(fi_df.head(30))

## 12. Báo cáo generalization và audit

In [ ]:
for report_path in [
    RESULTS_DIR / "generalization_report.md",
    RESULTS_DIR / "comparison_report.md",
    AUDIT_DIR / "audit_summary.md",
]:
    if report_path.exists():
        display(Markdown(f"## {report_path.name}"))
        display(Markdown(report_path.read_text(encoding="utf-8", errors="ignore")))
    else:
        print("Missing report:", report_path)

## 13. Live IPS events

Nếu đã chạy `python live_ips.py ...`, log thật sẽ nằm ở `results/live_events.csv`.

In [ ]:
live_events_path = RESULTS_DIR / "live_events.csv"
if live_events_path.exists():
    live_events = pd.read_csv(live_events_path)
    display(live_events.tail(100))

    if not live_events.empty and "prediction" in live_events.columns:
        plt.figure(figsize=(10, 5))
        sns.countplot(data=live_events, y="prediction", order=live_events["prediction"].value_counts().index)
        plt.title("Live IPS Event Prediction Counts")
        plt.tight_layout()
        plt.show()
else:
    print("No live events found. Run live_ips.py first.")

## 14. Model đã lưu

In [ ]:
model_files = sorted(MODELS_DIR.glob("*.pkl"))
display(pd.DataFrame({
    "model_file": [p.name for p in model_files],
    "size_mb": [p.stat().st_size / 1024 / 1024 for p in model_files],
}))

selected_model_path = MODELS_DIR / "selected_model.pkl"
print("selected_model.pkl exists:", selected_model_path.exists())

## 15. Lệnh chạy project

```powershell
# Train lại toàn bộ pipeline
python main.py

# Replay IDS/IPS demo không cần card mạng
python replay_ips.py --speed 0.2 --limit 500

# Chạy dashboard
streamlit run app.py

# Liệt kê interface bắt gói
python live_ips.py --list-interfaces

# Chạy live IPS simulation mode
python live_ips.py --interface "<interface-name>" --threshold 0.95

# Chạy Jupyter Lab
jupyter lab
```